In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib

In [2]:
df = pd.read_csv("master_features_advanced_1min.csv")

# Define Inputs (X) and Output (y)
X = df[['Heart_Rate', 'Ratio_R', 'Red_Variability', 'IR_Variability', 'Red_Slope', 'IR_Slope', 'Red_Skew', 'IR_Skew']]
y = df['Glucose']

# 2. SPLIT THE DATA
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. SCALE THE DATA
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
num_predictors = X_train.shape[1]
fine_gamma = 8.0 / num_predictors
medium_gamma = 0.5 / num_predictors

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5),
    
    "Support Vector Machine (SVR)": SVR(kernel='rbf', C=100, epsilon=0.1),
    "SVR-Fine Gaussian": SVR(kernel='rbf', gamma=fine_gamma, C=10.0, epsilon=0.1),
    "Non-Linear Medium Gaussian SVR": SVR(kernel='rbf', gamma=medium_gamma, C=10.0, epsilon=0.1),
    
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    "AdaBoost": AdaBoostRegressor(n_estimators=100, random_state=42),
    
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1),
    "CatBoost Regressor": CatBoostRegressor(iterations=300, depth=4, learning_rate=0.05, verbose=0, random_seed=42)
}

In [4]:
best_rmse = float('inf')
best_model = None
best_model_name = None

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    
    print(f"\n{name}:")
    print(f"  ➡ MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.2f}")
    
    if rmse < best_rmse:
        best_rmse = rmse
        best_model = model
        best_model_name = name


Linear Regression:
  ➡ MAE: 24.97 | RMSE: 33.27 | R2: 0.15

Ridge Regression:
  ➡ MAE: 24.64 | RMSE: 33.15 | R2: 0.15

Lasso Regression:
  ➡ MAE: 24.22 | RMSE: 32.73 | R2: 0.18

ElasticNet:
  ➡ MAE: 24.79 | RMSE: 33.21 | R2: 0.15

K-Nearest Neighbors:
  ➡ MAE: 23.13 | RMSE: 32.81 | R2: 0.17

Support Vector Machine (SVR):
  ➡ MAE: 17.67 | RMSE: 28.19 | R2: 0.39

SVR-Fine Gaussian:
  ➡ MAE: 24.13 | RMSE: 34.61 | R2: 0.08

Non-Linear Medium Gaussian SVR:
  ➡ MAE: 22.85 | RMSE: 34.07 | R2: 0.11

Random Forest:
  ➡ MAE: 23.13 | RMSE: 33.64 | R2: 0.13

Gradient Boosting:
  ➡ MAE: 28.35 | RMSE: 37.42 | R2: -0.08

AdaBoost:
  ➡ MAE: 23.87 | RMSE: 32.78 | R2: 0.17

XGBoost:
  ➡ MAE: 24.46 | RMSE: 33.01 | R2: 0.16


c:\Users\Revios\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



LightGBM:
  ➡ MAE: 24.92 | RMSE: 31.99 | R2: 0.21

CatBoost Regressor:
  ➡ MAE: 25.67 | RMSE: 33.89 | R2: 0.12


In [5]:
joblib.dump(best_model, 'SVR.joblib')
joblib.dump(scaler, 'scaler_SVR.joblib')

print(f"\nBest Model: {best_model_name} (RMSE: {best_rmse:.2f})")
print("Saved best model to 'SVR.joblib'")


Best Model: Support Vector Machine (SVR) (RMSE: 28.19)
Saved best model to 'SVR.joblib'
